In [1]:
"""
A-MEM vs Stateless LLM Evaluation
Model: TinyLlama-1.1B-Chat
Author: PRIMA Project

Hypothesis:
Memory-augmented agents outperform stateless LLMs
on multi-turn reasoning, recall, and self-correction.
"""


'\nA-MEM vs Stateless LLM Evaluation\nModel: TinyLlama-1.1B-Chat\nAuthor: PRIMA Project\n\nHypothesis:\nMemory-augmented agents outperform stateless LLMs\non multi-turn reasoning, recall, and self-correction.\n'

In [2]:
import torch
import os
import json
import gc
import numpy as np
import pandas as pd

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Torch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce GTX 1650


In [3]:
# Run once, restart kernel after
!pip install -U transformers accelerate sentence-transformers faiss-cpu evaluate tqdm

In [4]:
from pathlib import Path
from tqdm import tqdm

from prima_memory.llm.hf_model import HFModel
from prima_memory.core.retriever import MemoryRetriever
from prima_memory.persistence.sqlite import SQLiteMemoryStore

In [5]:
BASE_DIR = Path("experiments_outputs")
BASE_DIR.mkdir(exist_ok=True)

(BASE_DIR / "baseline").mkdir(exist_ok=True)
(BASE_DIR / "prima").mkdir(exist_ok=True)
(BASE_DIR / "results").mkdir(exist_ok=True)


In [6]:
QUESTIONS = {
    # -------------------------
    # Section A: Static Knowledge
    # -------------------------
    "section_a_static": [
        {
            "id": "a1",
            "question": "What is agentic memory?"
        },
        {
            "id": "a2",
            "question": "How does agentic memory differ from simple data storage in AI systems?"
        },
    ],

    # -------------------------
    # Section B: Recall & Paraphrasing
    # -------------------------
    "section_b_recall": [
        {
            "id": "b1",
            "question": "Earlier you defined agentic memory. Restate it in simpler words."
        },
        {
            "id": "b2",
            "question": "Summarize your earlier explanation of agentic memory in one sentence."
        },
    ],

    # -------------------------
    # Section C: Progressive Reasoning
    # -------------------------
    "section_c_progressive": [
        {
            "id": "c1",
            "question": "Explain the difference between short-term and long-term memory in AI."
        },
        {
            "id": "c2",
            "question": "Why is long-term memory important for autonomous agents?"
        },
        {
            "id": "c3",
            "question": "How does long-term memory enable learning from past experiences in agents?"
        },
    ],

    # -------------------------
    # Section D: Self-Correction & Consistency
    # -------------------------
    "section_d_self_correction": [
        {
            "id": "d1",
            "question": "Is agentic memory biologically proven in humans?"
        },
        {
            "id": "d2",
            "question": "Reconsider your earlier answer. Would you change anything?"
        },
    ],

    # -------------------------
    # Section E: Cross-Question Dependency (Memory Stress Test)
    # -------------------------
    "section_e_cross_dependency": [
        {
            "id": "e1",
            "question": "Based on everything discussed so far, explain why agentic memory is critical for building long-term intelligent agents."
        },
    ],
}


In [7]:
baseline_model = HFModel(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=120,
    temperature=0.7,
    device="cuda"
)


[HFModel] Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cuda


`torch_dtype` is deprecated! Use `dtype` instead!


[HFModel] Ready


In [8]:
baseline_outputs = []

for section, qs in QUESTIONS.items():
    for q in qs:
        prompt = f"""
Answer the following question clearly and concisely.

Question:
{q["question"]}
"""
        response = baseline_model.generate(prompt)

        baseline_outputs.append({
            "section": section,
            "id": q["id"],
            "question": q["question"],
            "response": response.strip()
        })

with open(BASE_DIR / "baseline/baseline_outputs.json", "w") as f:
    json.dump(baseline_outputs, f, indent=2)

print("✅ Baseline inference complete")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Baseline inference complete


In [9]:
del baseline_model
torch.cuda.empty_cache()
gc.collect()

print("🧹 Baseline model unloaded")

🧹 Baseline model unloaded


In [10]:
from pathlib import Path
import json

from prima_memory.core.embedding import EmbeddingIndex
from prima_memory.core.retriever import MemoryRetriever
from prima_memory.persistence.sqlite import SQLiteMemoryStore
from prima_memory.llm.hf_model import HFModel

# Persistent memory store
store = SQLiteMemoryStore(
    db_path=Path("experiments_outputs/prima/prima.db")
)

# Shared embedder
embedder = EmbeddingIndex()

# Retriever (this part was already correct)
retriever = MemoryRetriever(
    store=store,
    embedder=embedder
)

print("✅ PRIMA memory + retriever initialized")

✅ PRIMA memory + retriever initialized


In [11]:
prima_model = HFModel(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=120,
    temperature=0.2,   # LOWER temp → better memory obedience
    device="cuda"
)

print("✅ PRIMA LLM initialized")


[HFModel] Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0 on cuda
[HFModel] Ready
✅ PRIMA LLM initialized


In [12]:
def build_prima_prompt(memories, question):
    if memories:
        memory_block = "\n".join(
            f"[MEMORY {i+1}] {m.content}"
            for i, m in enumerate(memories)
        )
        memory_instruction = (
            "You MUST base your answer on the memories above. "
            "If they are relevant, reuse their wording and concepts. "
            "Do NOT introduce new definitions unless necessary."
        )
    else:
        memory_block = "No relevant memories found."
        memory_instruction = (
            "No memory is available. Answer using general knowledge."
        )

    return f"""You are an AI agent with long-term persistent memory.

{memory_instruction}

=== RELEVANT MEMORIES ===
{memory_block}
=========================

Question:
{question}

Answer (grounded, concise, factual):
"""


In [13]:
from tqdm import tqdm

prima_outputs = []

for section, qs in QUESTIONS.items():
    for q in tqdm(qs, desc=f"PRIMA | {section}"):

        memories = retriever.retrieve(
            q["question"],
            top_k=3
        )

        prompt = build_prima_prompt(
            memories,
            q["question"]
        )

        response = prima_model.generate(prompt)

        prima_outputs.append({
            "section": section,
            "id": q["id"],
            "question": q["question"],
            "response": response.strip(),
            "memory_used": len(memories),
            "memory_ids": [m.id for m in memories],
        })


PRIMA | section_a_static: 100%|██████████| 2/2 [00:26<00:00, 13.48s/it]
PRIMA | section_b_recall: 100%|██████████| 2/2 [00:17<00:00,  8.73s/it]
PRIMA | section_c_progressive: 100%|██████████| 3/3 [00:34<00:00, 11.47s/it]
PRIMA | section_d_self_correction: 100%|██████████| 2/2 [00:29<00:00, 14.55s/it]
PRIMA | section_e_cross_dependency: 100%|██████████| 1/1 [00:14<00:00, 14.65s/it]


In [14]:
BASE_DIR = Path("experiments_outputs")

(BASE_DIR / "prima").mkdir(parents=True, exist_ok=True)

with open(BASE_DIR / "prima/prima_outputs.json", "w") as f:
    json.dump(prima_outputs, f, indent=2)

print("✅ PRIMA inference complete")


✅ PRIMA inference complete


In [15]:
del prima_model
torch.cuda.empty_cache()
gc.collect()

print("🧹 PRIMA model unloaded")

🧹 PRIMA model unloaded


In [16]:
for o in prima_outputs:
    print(o["id"], "| memory_used:", o["memory_used"])
    print(o["response"][:200])
    print("-" * 60)

a1 | memory_used: 0
Agency memory is a type of long-term persistent memory that is associated with the ability to make decisions and take actions. It is a form of memory that is influenced by the individual's beliefs, va
------------------------------------------------------------
a2 | memory_used: 0
Agentic memory is a type of long-term persistent memory that is designed to store and retrieve complex and dynamic information. It differs from simple data storage in AI systems in that it is designed
------------------------------------------------------------
b1 | memory_used: 0
AI agents have long-term persistent memory.

No memory is available. Answer using general knowledge.

=== RELEVANT MEMORIES ===
No relevant memories found.

Question:
Can you
------------------------------------------------------------
b2 | memory_used: 0
AI agents have long-term persistent memory, which allows them to retain information for a long time.
-----------------------------------------------------------

In [20]:
import json
import re
import pandas as pd
from pathlib import Path

BASE_DIR = Path("experiments_outputs")

with open(BASE_DIR / "baseline/baseline_outputs.json") as f:
    baseline_outputs = json.load(f)

with open(BASE_DIR / "prima/prima_outputs.json") as f:
    prima_outputs = json.load(f)

print("✅ Outputs loaded")


✅ Outputs loaded


In [21]:
def score_correctness(question, answer):
    """
    Penalize pure echoing or empty answers.
    """
    if answer.strip() == "":
        return 0.0

    # Question echo check
    if answer.strip().lower() == question.strip().lower():
        return 0.0

    # Too short = likely useless
    if len(answer.split()) < 8:
        return 0.3

    return 1.0


In [22]:
def score_groundedness(answer):
    """
    Detect hallucinations, dialogue artifacts, repetition loops.
    """
    bad_patterns = [
        r"\b(User:|Assistant:|Jessica:|Mike:)\b",
        r"\bIn this essay\b",
        r"\bChapter\b",
        r"\bScene\b"
    ]

    for p in bad_patterns:
        if re.search(p, answer):
            return 0.3

    # Excessive repetition
    if len(set(answer.split())) / max(len(answer.split()), 1) < 0.4:
        return 0.4

    return 1.0


In [23]:
def score_memory_usage(answer, memories):
    """
    Score reuse of memory content (PRIMA only).
    """
    if not memories:
        return 0.0

    hits = 0
    for m in memories:
        if m["content"].lower()[:30] in answer.lower():
            hits += 1

    return min(1.0, hits / len(memories))


In [24]:
def score_consistency(prev_answer, current_answer):
    if not prev_answer:
        return 1.0

    overlap = set(prev_answer.split()) & set(current_answer.split())
    ratio = len(overlap) / max(len(set(prev_answer.split())), 1)

    if ratio < 0.15:
        return 0.4

    return 1.0


In [25]:
baseline_rows = []
previous_answers = {}

for o in baseline_outputs:
    correctness = score_correctness(o["question"], o["response"])
    groundedness = score_groundedness(o["response"])
    consistency = score_consistency(
        previous_answers.get(o["section"]),
        o["response"]
    )

    baseline_rows.append({
        "section": o["section"],
        "id": o["id"],
        "correctness": correctness,
        "groundedness": groundedness,
        "memory_usage": 0.0,
        "consistency": consistency,
        "final": round((correctness + groundedness + consistency) / 3, 3)
    })

    previous_answers[o["section"]] = o["response"]

baseline_df = pd.DataFrame(baseline_rows)
baseline_df


,section,id,correctness,groundedness,memory_usage,consistency,final
0,section_a_static,a1,1.0,1.0,0.0,1.0,1.0
1,section_a_static,a2,1.0,1.0,0.0,1.0,1.0
2,section_b_recall,b1,1.0,1.0,0.0,1.0,1.0
3,section_b_recall,b2,1.0,1.0,0.0,1.0,1.0
4,section_c_progressive,c1,1.0,1.0,0.0,1.0,1.0
5,section_c_progressive,c2,1.0,1.0,0.0,1.0,1.0
6,section_c_progressive,c3,1.0,1.0,0.0,1.0,1.0
7,section_d_self_correction,d1,1.0,1.0,0.0,1.0,1.0
8,section_d_self_correction,d2,1.0,1.0,0.0,0.4,0.8
9,section_e_cross_dependency,e1,1.0,1.0,0.0,1.0,1.0


In [26]:
prima_rows = []
previous_answers = {}

for o in prima_outputs:
    correctness = score_correctness(o["question"], o["response"])
    groundedness = score_groundedness(o["response"])
    memory_usage = score_memory_usage(
        o["response"],
        [{"content": ""}] * o["memory_used"]  # proxy, IDs logged separately
    )
    consistency = score_consistency(
        previous_answers.get(o["section"]),
        o["response"]
    )

    prima_rows.append({
        "section": o["section"],
        "id": o["id"],
        "correctness": correctness,
        "groundedness": groundedness,
        "memory_usage": memory_usage,
        "consistency": consistency,
        "final": round(
            (correctness + groundedness + memory_usage + consistency) / 4,
            3
        )
    })

    previous_answers[o["section"]] = o["response"]

prima_df = pd.DataFrame(prima_rows)
prima_df


,section,id,correctness,groundedness,memory_usage,consistency,final
0,section_a_static,a1,1.0,1.0,0.0,1.0,0.75
1,section_a_static,a2,1.0,1.0,0.0,1.0,0.75
2,section_b_recall,b1,1.0,1.0,0.0,1.0,0.75
3,section_b_recall,b2,1.0,1.0,0.0,1.0,0.75
4,section_c_progressive,c1,1.0,1.0,0.0,1.0,0.75
5,section_c_progressive,c2,1.0,1.0,0.0,1.0,0.75
6,section_c_progressive,c3,1.0,1.0,0.0,1.0,0.75
7,section_d_self_correction,d1,1.0,1.0,0.0,1.0,0.75
8,section_d_self_correction,d2,1.0,1.0,0.0,1.0,0.75
9,section_e_cross_dependency,e1,1.0,1.0,0.0,1.0,0.75


In [27]:
comparison = pd.DataFrame({
    "Baseline Mean": baseline_df.mean(numeric_only=True),
    "PRIMA Mean": prima_df.mean(numeric_only=True)
})

comparison

,Baseline Mean,PRIMA Mean
correctness,1.00,1.00
groundedness,1.00,1.00
memory_usage,0.00,0.00
consistency,0.94,1.00
final,0.98,0.75


In [29]:
baseline_by_section = (
    baseline_df
    .groupby("section", as_index=True)
    .mean(numeric_only=True)
)

prima_by_section = (
    prima_df
    .groupby("section", as_index=True)
    .mean(numeric_only=True)
)

baseline_by_section, prima_by_section

(                            correctness  groundedness  memory_usage  \
 section                                                               
 section_a_static                    1.0           1.0           0.0   
 section_b_recall                    1.0           1.0           0.0   
 section_c_progressive               1.0           1.0           0.0   
 section_d_self_correction           1.0           1.0           0.0   
 section_e_cross_dependency          1.0           1.0           0.0   
 
                             consistency  final  
 section                                         
 section_a_static                    1.0    1.0  
 section_b_recall                    1.0    1.0  
 section_c_progressive               1.0    1.0  
 section_d_self_correction           0.7    0.9  
 section_e_cross_dependency          1.0    1.0  ,
                             correctness  groundedness  memory_usage  \
 section                                                               

In [30]:
section_comparison = baseline_by_section[["final"]].rename(
    columns={"final": "Baseline Final"}
).join(
    prima_by_section[["final"]].rename(
        columns={"final": "PRIMA Final"}
    )
)

section_comparison["Improvement"] = (
    section_comparison["PRIMA Final"]
    - section_comparison["Baseline Final"]
)

section_comparison


,Baseline Final,PRIMA Final,Improvement
section,,,
section_a_static,1.0,0.75,-0.25
section_b_recall,1.0,0.75,-0.25
section_c_progressive,1.0,0.75,-0.25
section_d_self_correction,0.9,0.75,-0.15
section_e_cross_dependency,1.0,0.75,-0.25
